# CoT 截断 / 篡改实验：trace 到底有没有在起作用

## 要回答什么

模型写出的 CoT **是不是因果负载的**：

- 把 trace 从中间**截断**，它还能不能算对？
- 把中间某一步的值**改错**，它是「跟着错值继续走」还是「回头查表、自己纠正」？
- 这两个效应**随深度 d 怎么变**？

## 设计

对每道题（串行，`d = 5..20`，每档 40 题）先用 CoT 跑一次**干净**的输出，拿到 trace，
再基于这段 trace 构造三种干预：

| 条件 | 前缀（assistant 已经写了的部分） | 让模型继续做什么 |
|---|---|---|
| `clean` | 无（正常生成） | 写完 trace + 答案 |
| `trunc_cont` | 前 `t` 步（**正确状态**） | 接着写剩下的步骤 + 答案 |
| `trunc_force` | 前 `t` 步 + `####ANSWER=` | **不许再写步骤**，直接给答案 |
| `corrupt_cont` | 前 `t` 步，但第 `t` 步的值**改成 c** | 接着写剩下的步骤 + 答案 |

- `t = d // 2`（中间那一步）。
- `c` 选成「从 c 走完剩余步数得到的终点 ≠ gold」的第一个值，这样「跟着走」和「答对」可以区分。
- 续写用 **assistant 前缀 + `continue_final_message=True`**，模型不知道自己在被干预。

## 判分

| 指标 | 含义 |
|---|---|
| `answer_ok` | 最终答案 == gold |
| `follow` | 最终答案 == `implied`（从 c 按表走完剩余步数的终点）→ 被骗了 |
| `recover` | 最终答案 == gold → 无视篡改 / 回头查表 |
| `other` | 两个都不是 |
| `resume_ok` | 续写的第一个数字是否等于 `table[当前状态]`（诊断：有没有接着写，还是从头重来） |
| `restarted` | 续写第一个数字 == 起始值（从头重来） |

## 分析口径

**全量都跑，分析时分层**：主矩阵用全部题，另出一份「只含 clean 答对」的分层矩阵。
分层是必要的，因为 d 大时 clean 本身就有错，混在一起会把 `recover` 抬高（模型本来就不会做，
答对 gold 不代表它纠正了篡改）。

## 结果矩阵

`cot_metrics.json` 里的 `matrix` 是一个 **深 × 条件** 的表，每行一个深度 `d`，列的含义见 notebook
最后一节的说明表。同时写出 `cot_matrix.csv` 方便直接看。


## 1. 配置

- 深度 `d = 5..20`（16 档），每档 40 题。
- 每题的干预在**干净 CoT 解析成功**后才构造；解析失败的题记 `no_trace`，不构造干预。
- `DTYPE="bfloat16"`（4090）；T4 请改 `float16`。
- 采样参数不变：`temperature=0.6`、`top_p=0.95`。


In [ ]:
import json, os, re, sys, time, random
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd


def find_repo():
    here = Path.cwd()
    for p in [here] + list(here.parents):
        if (p / "Experiments" / "data_generation" / "synthesis_pointer_numeric_dataset.jsonl").exists():
            return p
    return here


REPO = find_repo()
DATA_DIR = REPO / "Experiments" / "data_generation"
OUT_DIR = REPO / "Experiments" / "final_experiments"
OUT_DIR.mkdir(parents=True, exist_ok=True)
DATASET = DATA_DIR / "synthesis_pointer_numeric_dataset.jsonl"

D_LIST = list(range(5, 21))
N_PER_DEPTH = 40
SEED = 20261112
TABLE_SIZE = 40

HF_ID = "Qwen/Qwen3-4B-Instruct-2507"
DTYPE = "bfloat16"
GPU_UTIL = 0.90
MAX_MODEL_LEN = 8192
MAX_NUM_SEQS = 32
TEMPERATURE, TOP_P = 0.6, 0.95

SMOKE = False
SMOKE_N = 3

OUT_CLEAN = OUT_DIR / "cot_clean.jsonl"
OUT_INTERV = OUT_DIR / "cot_interventions.jsonl"
OUT_METRICS = OUT_DIR / "cot_metrics.json"
OUT_CSV = OUT_DIR / "cot_matrix.csv"

print("REPO:", REPO)
print("depths:", D_LIST, "| %d 题/档 -> %d 题" % (N_PER_DEPTH, len(D_LIST) * N_PER_DEPTH))
print("预估生成量: clean %d + 干预 %d = %d" % (
    len(D_LIST) * N_PER_DEPTH, len(D_LIST) * N_PER_DEPTH * 3, len(D_LIST) * N_PER_DEPTH * 4))
print("dtype:", DTYPE)


## 2. 读题

串行题，`d = 5..20`，每档 40 题。每题保留题干、gold、40 行表（转成 dict 方便走路）。


In [ ]:
pool = defaultdict(list)
with open(DATASET, encoding="utf-8") as f:
    for line in f:
        r = json.loads(line)
        if r["kind"] != "serial" or r["d"] not in D_LIST:
            continue
        pool[r["d"]].append({
            "qid": r["qid"],
            "d": r["d"],
            "q": r["q"],
            "gold": int(r["ans"]),
            "start": int(r["chains"][0]["initial"]),
            "table": {row["index"]: row["value"] for row in r["table"]},
        })

QUESTIONS = []
for d in D_LIST:
    rng = random.Random("%s|d=%d" % (SEED, d))
    picked = rng.sample(pool[d], min(N_PER_DEPTH, len(pool[d])))
    picked.sort(key=lambda x: x["qid"])
    QUESTIONS.extend(picked)
    print("d=%2d: 可用 %3d -> 抽 %2d" % (d, len(pool[d]), len(picked)))

assert len(QUESTIONS) == N_PER_DEPTH * len(D_LIST), len(QUESTIONS)
QM = {q["qid"]: q for q in QUESTIONS}


def walk(tbl, start, steps):
    v = start
    for _ in range(steps):
        v = tbl[v]
    return v


# 自检：从 start 走 d 步 == gold
for q in QUESTIONS[:50]:
    assert walk(q["table"], q["start"], q["d"]) == q["gold"], q["qid"]
print("\nsanity: walk(start, d) == gold 通过")


## 3. Prompt

沿用 task A 的 CoT 前导语（单链版），没有 keyword 行约束（k=1）。

对照一下你这个任务上模型的真实输出格式：**裸数字，一行一个**，第一行是起点，最后一行是最终值，
然后 `####ANSWER=<int>`。所以截断点 = 第 `t` 个数字之后，篡改 = 换掉第 `t` 个数字。


In [ ]:
COT_PREAMBLE = ("Solve the problem step by step. Before the final answer line, explicitly write the index "
                "reached after each pointer step, one per line. The keyword lines required by the rules "
                "below must come after those steps and before the final answer line.\n")

BASE_RULE = ("Output format. Your response must contain only the final answer line after the steps. The "
             "answer line is always the very last line of the response: ####ANSWER= followed by the final "
             "index as an integer, and nothing may be written after it. Do not repeat or restate these rules.")

SYSTEM_COT = COT_PREAMBLE + "Rules:\nR1. " + BASE_RULE


def build_clean_messages(q):
    return [{"role": "system", "content": SYSTEM_COT}, {"role": "user", "content": q["q"]}]


def build_prefix_messages(q, prefix):
    return build_clean_messages(q) + [{"role": "assistant", "content": prefix}]


print(SYSTEM_COT[:400])


## 4. 加载 vLLM

非 thinking 模型，不传 `enable_thinking`。4090 用 bf16。


In [ ]:
import torch
from vllm import LLM, SamplingParams

if not torch.cuda.is_available():
    raise RuntimeError("没有 CUDA。请在 GPU 机器上执行这一格。")

cap = torch.cuda.get_device_capability(0)
mem_gib = torch.cuda.get_device_properties(0).total_memory / 2**30
if DTYPE == "bfloat16" and cap[0] < 8:
    raise RuntimeError("这张卡不支持 bf16（capability %s），请把 DTYPE 改成 'float16'。" % (cap,))

llm = LLM(
    model=HF_ID,
    dtype=DTYPE,
    gpu_memory_utilization=GPU_UTIL,
    max_model_len=MAX_MODEL_LEN,
    max_num_seqs=MAX_NUM_SEQS,
)

SP_CLEAN = SamplingParams(temperature=TEMPERATURE, top_p=TOP_P, max_tokens=2048)
SP_CONT = SamplingParams(temperature=TEMPERATURE, top_p=TOP_P, max_tokens=1024)
print("GPU: %.1f GiB, capability %s | torch %s cuda %s" % (mem_gib, cap, torch.__version__, torch.version.cuda))


## 5. 阶段 1：干净 CoT

每题跑一次，保存 system / user / output。断点续跑。

这一步的输出同时是后面所有干预的**原料**：从里面解析出 `[起点, 第1步, 第2步, …]` 的数值序列。


In [ ]:
MARK = "####ANSWER="


def load_done(path):
    if not path.exists():
        return set()
    with open(path, encoding="utf-8") as f:
        return {json.loads(l)["id"] for l in f if l.strip()}


def run_clean(questions, path=OUT_CLEAN):
    done = load_done(path)
    todo = [q for q in questions if q["qid"] not in done]
    print("clean: 总 %d，已完成 %d，待跑 %d" % (len(questions), len(questions) - len(todo), len(todo)))
    for s in range(0, len(todo), 32):
        chunk = todo[s:s + 32]
        msgs = [build_clean_messages(q) for q in chunk]
        t0 = time.time()
        outs = llm.chat(msgs, SP_CLEAN, use_tqdm=False)
        dt = time.time() - t0
        with open(path, "a", encoding="utf-8") as f:
            for q, o in zip(chunk, outs):
                c = o.outputs[0]
                f.write(json.dumps({
                    "id": q["qid"], "qid": q["qid"], "d": q["d"], "gold": q["gold"], "start": q["start"],
                    "output": c.text, "n_tokens": len(c.token_ids),
                    "finish_reason": c.finish_reason, "trunc": c.finish_reason == "length",
                    "prompt_tokens": len(o.prompt_token_ids), "batch_sec": round(dt, 2),
                    "system": SYSTEM_COT, "user": q["q"],
                }, ensure_ascii=False) + "\n")
        print("[clean] %d/%d  %.1fs" % (s + len(chunk), len(todo), dt))


if SMOKE:
    run_clean(QUESTIONS[: SMOKE_N * 2])
else:
    run_clean(QUESTIONS)


## 6. trace 解析

真实格式是裸数字。解析规则：

1. 逐行看，遇到 `####ANSWER=` 停；
2. 每行必须是纯整数（允许 `Step 3: 17` 这种前缀，容错）；
3. 得到的列表第 0 个是起点，第 k 个是走完 k 步后的值。

解析失败（有非数字行）的题记 `no_trace`，不构造干预。


In [ ]:
def parse_trace(text):
    vals = []
    for line in str(text).splitlines():
        s = line.strip()
        if not s:
            continue
        if s.startswith(MARK):
            break
        m = re.fullmatch(r"(?:[Ss]tep\s*\d+\s*[:.)\-]?\s*)?(-?\d+)", s)
        if not m:
            return None
        vals.append(int(m.group(1)))
    return vals


def parse_answer(text):
    lines = [l.strip() for l in str(text).splitlines() if l.strip()]
    idx = [i for i, l in enumerate(lines) if l.startswith(MARK)]
    if len(idx) == 0:
        return None, "no_marker"
    if len(idx) > 1:
        return None, "multi_marker"
    i = idx[0]
    raw = lines[i][len(MARK):].strip()
    if raw == "" and i + 1 < len(lines) and re.fullmatch(r"-?\d+", lines[i + 1]):
        raw = lines[i + 1]
        i = i + 1
    if i != len(lines) - 1:
        return None, "not_last"
    if not re.fullmatch(r"-?\d+", raw):
        return None, "not_int"
    return int(raw), "ok"


# ---- 解析干净输出 ----
clean_rows = [json.loads(l) for l in open(OUT_CLEAN, encoding="utf-8") if l.strip()]
TRACE = {}
stats = Counter()
for r in clean_rows:
    q = QM[r["qid"]]
    vals = parse_trace(r["output"])
    ans, why = parse_answer(r["output"])
    r["trace"] = vals
    r["answer"] = ans
    r["answer_reason"] = why
    r["answer_ok"] = (why == "ok" and ans == q["gold"])
    r["trace_ok"] = vals is not None and len(vals) >= 3
    stats["n"] += 1
    stats["answer_ok"] += int(r["answer_ok"])
    stats["trace_ok"] += int(r["trace_ok"])
    stats["len_%d" % (len(vals) if vals else -1)] += 1
    if r["trace_ok"]:
        TRACE[r["qid"]] = vals

print("clean:", stats["n"], "| answer_ok %.3f | trace_ok %.3f" % (
    stats["answer_ok"] / stats["n"], stats["trace_ok"] / stats["n"]))
EX = Counter()
for r in clean_rows[:200]:
    if r["trace_ok"]:
        EX[len(r["trace"]) - 1 - r["d"]] += 1
print("trace 长度 - d - 1 的分布（0 = 正好 d 步）:", dict(sorted(EX.items())))
CLEAN_BY_QID = {r["qid"]: r for r in clean_rows}
print("\n示例:", [(r["qid"][:8], r["trace"][:3], "...", len(r["trace"])) for r in clean_rows[:2]])


## 7. 构造三种干预

对每个有 trace 的题（`vals` 长度 ≥ 2）：

```
t = min(d // 2, len(vals) - 1)      # 中间那一步
v = vals[t]                          # 这一步的正确值
c = 第一个 (v+delta)%40 使 walk(c, d-t) != gold   # 篡改值
```

三个前缀：

| 条件 | 前缀 | 结束处 |
|---|---|---|
| `trunc_cont` | `vals[0..t]` | 停在第 t 步的值后面 |
| `trunc_force` | `vals[0..t]` + `\n####ANSWER=` | 强制立刻答 |
| `corrupt_cont` | `vals[0..t-1] + [c]` | 停在错误值后面 |

`implied = walk(c, d - t)`；`expected_next = table[vals[t]]`（用于 `resume_ok`）。


In [ ]:
PLAN = {}
for qid, vals in TRACE.items():
    q = QM[qid]
    d, tbl, gold = q["d"], q["table"], q["gold"]
    t = min(d // 2, len(vals) - 1)
    if t < 1 or t >= len(vals):
        continue
    v = vals[t]
    c = None
    implied = None
    for delta in range(1, TABLE_SIZE):
        cand = (v + delta) % TABLE_SIZE
        if cand == v:
            continue
        imp = walk(tbl, cand, d - t)
        if imp != gold:
            c, implied = cand, imp
            break
    if c is None:
        continue
    PLAN[qid] = {
        "t": t, "v": v, "c": c, "implied": implied,
        "prefix_trunc": "\n".join(str(x) for x in vals[: t + 1]),
        "prefix_corrupt": "\n".join(str(x) for x in vals[:t] + [c]),
        "expected_next": tbl[v],
        "next_after_corrupt": tbl[c],
        "remaining": d - t,
        "trace_len": len(vals),
    }

print("可构造干预的题: %d / %d" % (len(PLAN), len(TRACE)))
assert len(PLAN) > 0
ex = list(PLAN.items())[0]
print("示例 qid=%s d=%d t=%d v=%d -> c=%d implied=%d" % (
    ex[0][:10], QM[ex[0]]["d"], ex[1]["t"], ex[1]["v"], ex[1]["c"], ex[1]["implied"]))
print("trunc 前缀:\n" + ex[1]["prefix_trunc"])
print("corrupt 前缀:\n" + ex[1]["prefix_corrupt"])


## 8. 探针：确认 `continue_final_message` 可用

一次很短的调用，验证当前 vLLM 支持 assistant 前缀续写。失败的话这一格会给出明确报错，
而不是跑到一半才崩。


In [ ]:
_probe = [
    {"role": "system", "content": "You are a calculator. Answer with a single integer."},
    {"role": "user", "content": "2 + 2 ="},
    {"role": "assistant", "content": "####ANSWER="},
]
try:
    _o = llm.chat(_probe, SamplingParams(temperature=0.0, max_tokens=16),
                  use_tqdm=False, continue_final_message=True, add_generation_prompt=False)
    print("probe ok ->", repr(_o[0].outputs[0].text))
except TypeError as e:
    raise RuntimeError(
        "当前 vLLM 不支持 continue_final_message / add_generation_prompt。\n"
        "请升级 vLLM，或改用「把前缀放进 user 消息」的替代方案。\n原始错误: %r" % (e,))


## 9. 阶段 2：跑干预

每题 3 条（`trunc_cont` / `trunc_force` / `corrupt_cont`），全部用 assistant 前缀续写。
生成的 completion 和 prefix 分开存，判分时拼起来解析。


In [ ]:
def run_interv(plans, path=OUT_INTERV):
    done = load_done(path)
    jobs = []
    for qid, p in plans.items():
        q = QM[qid]
        jobs.append((qid, "trunc_cont", p["prefix_trunc"]))
        jobs.append((qid, "trunc_force", p["prefix_trunc"] + "\n" + MARK))
        jobs.append((qid, "corrupt_cont", p["prefix_corrupt"]))
    todo = [j for j in jobs if "%s|%s" % (j[0], j[1]) not in done]
    print("干预: 总 %d，已完成 %d，待跑 %d" % (len(jobs), len(jobs) - len(todo), len(todo)))

    for s in range(0, len(todo), 32):
        chunk = todo[s:s + 32]
        msgs = [build_prefix_messages(QM[qid], prefix) for qid, _, prefix in chunk]
        t0 = time.time()
        outs = llm.chat(msgs, SP_CONT, use_tqdm=False,
                        continue_final_message=True, add_generation_prompt=False)
        dt = time.time() - t0
        with open(path, "a", encoding="utf-8") as f:
            for (qid, cond, prefix), o in zip(chunk, outs):
                c = o.outputs[0]
                f.write(json.dumps({
                    "id": "%s|%s" % (qid, cond), "qid": qid, "cond": cond, "d": QM[qid]["d"],
                    "prefix": prefix, "completion": c.text,
                    "n_tokens": len(c.token_ids), "finish_reason": c.finish_reason,
                    "trunc": c.finish_reason == "length",
                    "batch_sec": round(dt, 2),
                }, ensure_ascii=False) + "\n")
        print("[interv] %d/%d  %.1fs" % (s + len(chunk), len(todo), dt))


if SMOKE:
    run_interv(dict(list(PLAN.items())[: SMOKE_N * 2]))
else:
    run_interv(PLAN)


## 10. 判分与结果矩阵

每条干预记录都拼成 `full = prefix + completion` 再解析：

- `answer_ok`：答案 == gold
- `follow`：答案 == `implied`（只在 `corrupt_cont` 上有意义）
- `recover`：答案 == gold（`corrupt_cont` 上表示「没被骗」）
- `other`：两者都不是
- `resume_ok`：续写第一个数字 == `expected_next`（诊断「有没有接着写」）
- `restarted`：续写第一个数字 == 起点值（从头重来）

矩阵行 = 深度 `d`，列 = 条件 × 指标。完整列说明在最后一节。


In [ ]:
clean_by_qid = CLEAN_BY_QID          # 来自第 6 节，已带 answer_ok / trace
clean_rows = list(clean_by_qid.values())
interv = [json.loads(l) for l in open(OUT_INTERV, encoding="utf-8") if l.strip()]

recs = []
for r in interv:
    qid, cond = r["qid"], r["cond"]
    q, p = QM[qid], PLAN[qid]
    full = r["prefix"] + r["completion"]
    ans, why = parse_answer(full)
    comp = parse_trace(r["completion"].replace(MARK, "\n"))
    first = comp[0] if comp else None

    exp_next = p["next_after_corrupt"] if cond == "corrupt_cont" else p["expected_next"]
    rec = {
        "qid": qid, "cond": cond, "d": q["d"], "gold": q["gold"],
        "answer": ans, "answer_reason": why,
        "answer_ok": bool(why == "ok" and ans == q["gold"]),
        "follow": bool(cond == "corrupt_cont" and why == "ok" and ans == p["implied"]),
        "recover": bool(cond == "corrupt_cont" and why == "ok" and ans == q["gold"]),
        "other": bool(cond == "corrupt_cont" and why == "ok"
                      and ans not in (q["gold"], p["implied"])),
        "implied": p["implied"], "t": p["t"], "v": p["v"], "c": p["c"],
        "first_num": first,
        "resume_ok": bool(first is not None and first == exp_next),
        "restarted": bool(first is not None and first == q["start"]),
        "n_steps_out": len(comp),
        "trunc": r["trunc"], "n_tokens": r["n_tokens"],
        "clean_answer_ok": bool(clean_by_qid.get(qid, {}).get("answer_ok", False)),
    }
    recs.append(rec)

adf = pd.DataFrame(recs)
print("干预记录:", len(adf), "| cond 分布:", adf["cond"].value_counts().to_dict())


# ---- 主矩阵（全量） ----
matrix = {}
for d in D_LIST:
    sub = adf[adf["d"] == d]
    if not len(sub):
        continue
    row = {}
    cl = [clean_by_qid[q] for q in PLAN if QM[q]["d"] == d and q in clean_by_qid]
    row["clean_answer_ok"] = float(np.mean([r["answer_ok"] for r in cl])) if cl else float("nan")
    for cond in ("trunc_cont", "trunc_force", "corrupt_cont"):
        g = sub[sub["cond"] == cond]
        row["%s_n" % cond] = int(len(g))
        row["%s_answer_ok" % cond] = float(g["answer_ok"].mean()) if len(g) else float("nan")
        if cond != "trunc_force":   # 强制作答没有「续写」，resume_ok 不适用
            row["%s_resume_ok" % cond] = float(g["resume_ok"].mean()) if len(g) else float("nan")
    g = sub[sub["cond"] == "corrupt_cont"]
    row["corrupt_follow"] = float(g["follow"].mean()) if len(g) else float("nan")
    row["corrupt_recover"] = float(g["recover"].mean()) if len(g) else float("nan")
    row["corrupt_other"] = float(g["other"].mean()) if len(g) else float("nan")
    matrix[d] = row

# ---- 分层矩阵（只用 clean 答对的题） ----
strat = {}
for d in D_LIST:
    sub = adf[(adf["d"] == d) & (adf["clean_answer_ok"])]
    if not len(sub):
        continue
    g = sub[sub["cond"] == "corrupt_cont"]
    strat[d] = {
        "n": int(len(sub)),
        "trunc_cont_answer_ok": float(sub[sub["cond"] == "trunc_cont"]["answer_ok"].mean()),
        "trunc_force_answer_ok": float(sub[sub["cond"] == "trunc_force"]["answer_ok"].mean()),
        "corrupt_follow": float(g["follow"].mean()) if len(g) else float("nan"),
        "corrupt_recover": float(g["recover"].mean()) if len(g) else float("nan"),
        "corrupt_other": float(g["other"].mean()) if len(g) else float("nan"),
    }

mat_df = pd.DataFrame(matrix).T
print("\n主矩阵（全量）:")
display(mat_df.round(3))
print("\n分层矩阵（clean 答对的题）:")
display(pd.DataFrame(strat).T.round(3))

mat_df.to_csv(OUT_CSV, encoding="utf-8")
with open(OUT_METRICS, "w", encoding="utf-8") as f:
    json.dump({
        "meta": {
            "model": HF_ID, "dtype": DTYPE, "depths": D_LIST, "n_per_depth": N_PER_DEPTH,
            "seed": SEED, "temperature": TEMPERATURE, "top_p": TOP_P,
            "t_rule": "t = min(d//2, len(vals)-1)",
            "corruption": "replace step t's value with the first c whose walked endpoint != gold",
            "n_clean": len(clean_rows), "n_interv": len(adf),
            "n_planned": len(PLAN),
        },
        "matrix": matrix,
        "matrix_strat_clean_correct": strat,
        "records": recs,
    }, f, ensure_ascii=False, indent=2)
print("\nwritten:", OUT_METRICS, "|", OUT_CSV)


## 11. 结果矩阵每一列是什么

`matrix` 的行是深度 `d`（5..20）。列：

### 基线

| 列 | 含义 |
|---|---|
| `clean_answer_ok` | 干净 CoT 的答案正确率。**这是所有干预的参照**，也是分层的筛选依据 |

### 截断后继续（`trunc_cont`）

| 列 | 含义 |
|---|---|
| `trunc_cont_n` | 该深度参与统计的题数 |
| `trunc_cont_answer_ok` | 停在正确的第 `t` 步、允许继续写，最终答案正确的比例。**接近 `clean` 就说明「截断本身不影响」**，也就说明前缀里的状态足够它接着算 |
| `trunc_cont_resume_ok` | 续写的第一个数字 == `table[第 t 步的值]` 的比例。低说明它在重写/重算，而不是接着写 |

### 截断后强制作答（`trunc_force`）

| 列 | 含义 |
|---|---|
| `trunc_force_n` | 题数 |
| `trunc_force_answer_ok` | 停在正确的第 `t` 步、**不许再写步骤**、直接给答案的正确率。**这一列才是「不写 CoT 还能不能算」**：它随 `d` 掉得多快，就是「没写出来的步骤有多重要」 |
| `trunc_force` 没有 `resume_ok` | 它没有「续写」这一步（前缀直接以 `####ANSWER=` 结尾），所以该列不计算 |

### 篡改后继续（`corrupt_cont`）

| 列 | 含义 |
|---|---|
| `corrupt_cont_n` | 题数 |
| `corrupt_cont_answer_ok` | 答案正确的比例（不管它走哪条路） |
| `corrupt_follow` | 答案 == `implied`（从篡改值 `c` 走完剩余步数的终点）→ **被骗了，trace 在起作用** |
| `corrupt_recover` | 答案 == `gold` → **没被骗，回头查表 / 自己纠正了** |
| `corrupt_other` | 两个都不是（纯错） |
| `corrupt_cont_resume_ok` | 续写第一个数字 == `table[c]` 的比例（跟着错误状态走） |

### 分层矩阵 `matrix_strat_clean_correct`

只保留 `clean` 答对的题，列同上（少一些）。**主结论看这一份**：
- `corrupt_follow` 高 → 模型确实沿着写下的中间值走；
- `corrupt_recover` 高 → 它在重算，trace 只是装饰；
- `trunc_force_answer_ok` 随 `d` 下降 → 不写步骤就算不动，说明步骤本身是必要的。

### 怎么读这三条曲线

| 现象 | 解释 |
|---|---|
| `trunc_cont ≈ clean`，`trunc_force` 随 d 掉 | 前缀状态够用，但**剩余步骤不能省**——CoT 是做计算用的，不是复述 |
| `corrupt_follow` 随 d 上升 | 越深越依赖写下的 trace，篡改越容易传播 |
| `corrupt_recover` 恒高 | 模型并不真的沿 trace 走，答案另算（trace 不忠实） |
| `corrupt_other` 高 | 篡改让它混乱，既没跟随也没纠正 |


In [ ]:
for d in (5, 12, 20):
    sub = adf[(adf["d"] == d) & (adf["cond"] == "corrupt_cont") & (adf["clean_answer_ok"])]
    if not len(sub):
        continue
    print("=" * 100)
    print("d=%d corrupt_cont | n=%d follow=%.2f recover=%.2f other=%.2f" % (
        d, len(sub), sub["follow"].mean(), sub["recover"].mean(), sub["other"].mean()))
    for _, r in sub.head(2).iterrows():
        p = PLAN[r["qid"]]
        print("-" * 70)
        print("gold=%d implied=%d t=%d v=%d c=%d | answer=%s" % (
            r["gold"], p["implied"], p["t"], p["v"], p["c"], r["answer"]))
        print("prefix 末尾:", repr("\n".join(p["prefix_corrupt"].splitlines()[-3:])))
